In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

In [ ]:
load_dotenv(override=True)

openrouter_api_key = os.getenv('OPENROUTER_API_KEY')
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
ds_api_key = os.getenv('DEEPSEEK_API_KEY')
grok_api_key = os.getenv('GROK_API_KEY')

In [ ]:
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

# Consider using an enum for this.
MODEL_MAP = {
    "CLAUDE": {
        "model": "claude-3-5-sonnet-20240620",
        "api_key": anthropic_api_key,
        "endpoint": "https://api.anthropic.com/v1"
    },
    "GPT": {
        "model": "gpt-4o-mini",
        "api_key": openai_api_key,
        "endpoint": OPENROUTER_BASE_URL,
    },
    "Grok": {
        "model": "grok-beta",
        "api_key": grok_api_key,
        "endpoint": "https://api.grok.com/v1"
    },   
    "DeepSeek":{
        "model": "deepseek-reasoner",
        "api_key": ds_api_key,
        "endpoint": "https://api.deepseek.com/v1",
    },
    "Google": {
        "model": "gemini-2.0-flash-exp",
        "api_key": google_api_key,
        "endpoint": "https://generativelanguage.googleapis.com/v1beta/openai"
    },
}

In [ ]:
def call_openai(model, messages, temperature=0.7):
    client = OpenAI(
        base_url=MODEL_MAP[model]["endpoint"],
        api_key=MODEL_MAP[model]["api_key"]
    )
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature
    )
    return response.choices[0].message.content


In [ ]:
SYSTEM_PROMPT = """
You are an expert medical data scientist generating high-fidelity, synthetic patient health records for research purposes.
Your goal is to create data that mimics realistic, complex health patterns, including correlations between lifestyle, demographics, and clinical lab values.
The output must be in JSON or comma-separated CSV format.
Ensure that all data points are plausible (e.g., age 80 does not have a resting heart rate of 30, BMI of 40 correlates with high blood pressure, etc.).
Do not include any real patient identifiers (PHI).
Generate 20 distinct, uncorrelated features per record.
"""

USER_PROMPT = """
Role: Act as an expert data scientist and clinical informaticist specializing in generating high-fidelity, privacy-compliant synthetic healthcare data.
Task: Generate a CSV-formatted dataset of 100 synthetic patient records. 
Dataset Specifications:

    Target: Individuals' health conditions and associated metrics.
    Format: CSV, comma-separated.
    Privacy: No real PII (Personally Identifiable Information). Use realistic but artificial data.
    Data Types: Include a mix of numerical, categorical, and binary variables.
    Realism: Ensure logical correlations between features (e.g., higher BMI correlates with higher blood pressure, older age correlates with higher prevalence of chronic conditions). 

Features to Include (20 total):

    Patient_ID (Unique alphanumeric)
    Age (Integer, 18-90)
    Gender (Categorical: Male, Female, Other)
    Height_cm (Numerical)
    Weight_kg (Numerical)
    BMI (Calculated)
    BloodPressure_Systolic (Numerical)
    BloodPressure_Diastolic (Numerical)
    HeartRate_BPM (Numerical)
    Smoking_Status (Categorical: Never, Former, Current)
    Alcohol_Consumption (Categorical: Low, Medium, High)
    Physical_Activity_Hours_Per_Week (Numerical)
    HbA1c_Level (Numerical, 4.0 - 10.0)
    LDL_Cholesterol_mgdL (Numerical)
    Daily_Sleep_Hours (Numerical)
    Known_Allergies (Categorical: None, Penicillin, Peanuts, Pollen)
    Primary_Condition (Categorical: Hypertension, Type 2 Diabetes, Asthma, None)
    Number_of_Medications (Integer)
    Mental_Health_Score_PHQ9 (Integer, 0-27)
    Hospital_Readmission_30d (Binary: 0 or 1) 

Output Format:
Provide the output directly in CSV format inside a markdown code block. 
"""

messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": USER_PROMPT}
        ]